Imports

In [1]:
import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

import Bio.PDB
from Bio.PDB import PDBList
import numpy as np

import matplotlib.pyplot as plt


import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import torchmetrics


c:\Users\Adrian\Documents\Studium\M1\ai\AI-for-Chemistry-Project\ai forchem\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Dataset

In [2]:
df = pd.read_csv("Datasets/Fluorescent-Protein-Database.csv")


making PDB filenames

In [3]:
df["PDB_code"] = df["PDB code"]
#PDB_code more digestible for python as python despises spaces in strings

df["PDB_filename"] = df["PDB_code"].str.lower().apply(
    lambda x: f"pdbs/pdb{x}.ent"
)



Creating Contact Map Matrices using Biopython

In [4]:
def residue_dist(residue_one, residue_two) :
    """Returns the C-alpha distance between two residues"""
    diff_vector  = residue_one["CA"].coord - residue_two["CA"].coord
    return np.sqrt(np.sum(diff_vector * diff_vector))

def dist_matrix(chain_one, chain_two) :
    """Returns a matrix of C-alpha distances between two chains"""
    answer = np.zeros((len(chain_one), len(chain_two)), float)
    for row, residue_one in enumerate(chain_one) :
        for col, residue_two in enumerate(chain_two) :
            answer[row, col] = residue_dist(residue_one, residue_two)
    return answer

def get_distance_matrices(pdb_code, pdb_filename):
    parser = Bio.PDB.PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_code, pdb_filename)
    model = structure[0]

    chain = next(model.get_chains())

    residues = [r for r in chain if "CA" in r]

    return dist_matrix(residues, residues)

pdbl = PDBList()

for code in df["PDB_code"]:
    pdbl.retrieve_pdb_file(code, pdir="pdbs", file_format="pdb")



df["dist_matrix"] = df.apply(
    lambda row: get_distance_matrices(row["PDB_code"], row["PDB_filename"]),
    axis=1
)


Structure exists: 'pdbs\pdb3kct.ent' 
Structure exists: 'pdbs\pdb7oin.ent' 
Structure exists: 'pdbs\pdb8arm.ent' 
Structure exists: 'pdbs\pdb3nt3.ent' 
Structure exists: 'pdbs\pdb3u0l.ent' 
Structure exists: 'pdbs\pdb7ry2.ent' 
Structure exists: 'pdbs\pdb6u1a.ent' 
Structure exists: 'pdbs\pdb3wck.ent' 
Structure exists: 'pdbs\pdb3gb3.ent' 
Structure exists: 'pdbs\pdb1uis.ent' 
Structure exists: 'pdbs\pdb3e5t.ent' 
Structure exists: 'pdbs\pdb8xc6.ent' 
Structure exists: 'pdbs\pdb3ir8.ent' 
Structure exists: 'pdbs\pdb4nws.ent' 
Structure exists: 'pdbs\pdb7qgk.ent' 
Structure exists: 'pdbs\pdb3svn.ent' 
Structure exists: 'pdbs\pdb3nez.ent' 
Structure exists: 'pdbs\pdb3pj5.ent' 
Structure exists: 'pdbs\pdb3bxa.ent' 
Structure exists: 'pdbs\pdb3ned.ent' 
Structure exists: 'pdbs\pdb2qlg.ent' 
Structure exists: 'pdbs\pdb4edo.ent' 
Structure exists: 'pdbs\pdb3ip2.ent' 
Structure exists: 'pdbs\pdb4oqw.ent' 
Structure exists: 'pdbs\pdb4eds.ent' 
Structure exists: 'pdbs\pdb4kgf.ent' 
Structure ex

Matrices are preprocessed, as size must be consistent. This is ensured by submitting the given distance matrices to bilinear interpolation:

In [21]:
def preprocess(matrix, size=28):
    x = np.array(matrix, dtype=np.float32)
    x = torch.tensor(x)
    if x.ndim == 3:
        x = x.squeeze()
    x = x.unsqueeze(0).unsqueeze(0)
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)

    return x

df["preprocessed"] = df["dist_matrix"].apply(lambda x: preprocess(x))

A CNN is built and trained:

In [17]:
batch_size = 60

train_dataset = datasets.MNIST(root="dataset/", download=True, train=True, transform=transforms.ToTensor())
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = datasets.MNIST(root="dataset/", download=True, train=False, transform=transforms.ToTensor())
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

class CNN(nn.Module):
   def __init__(self, in_channels, num_classes):

       """
       Building blocks of convolutional neural network.

       Parameters:
           * in_channels: Number of channels in the input image (for grayscale images, 1)
           * num_classes: Number of classes to predict. In our problem, 10 (i.e digits from  0 to 9).
       """
       super(CNN, self).__init__()

       # 1st convolutional layer
       self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=3, padding=1)
       # Max pooling layer
       self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
       # 2nd convolutional layer
       self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
       # Fully connected layer
       self.fc1 = nn.Linear(16 * 7 * 7, num_classes)

   def forward(self, x):
    x = F.relu(self.conv1(x))
    x = self.pool(x)
    x = F.relu(self.conv2(x))
    x = self.pool(x)
    x = x.reshape(x.shape[0], -1)

    self.features = x   # <-- store embedding

    x = self.fc1(x)
    return x
       
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN(in_channels=1, num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs=10
for epoch in range(num_epochs):
 # Iterate over training batches
   print(f"Epoch [{epoch + 1}/{num_epochs}]")

   for batch_index, (data, targets) in enumerate(tqdm(train_loader)):
       data = data.to(device)
       targets = targets.to(device)
       scores = model(data)
       loss = criterion(scores, targets)
       optimizer.zero_grad()
       loss.backward()
       optimizer.step()



Epoch [1/10]


100%|██████████| 1000/1000 [00:10<00:00, 97.07it/s]


Epoch [2/10]


100%|██████████| 1000/1000 [00:09<00:00, 103.39it/s]


Epoch [3/10]


...

100%|██████████| 1000/1000 [00:09<00:00, 101.13it/s]


Epoch [9/10]


100%|██████████| 1000/1000 [00:10<00:00, 97.29it/s]


Epoch [10/10]


100%|██████████| 1000/1000 [00:10<00:00, 95.56it/s]


Now, the preprocessed distance matrice are passed trough the trained CNN, yielding vectors:

In [29]:
def encoder(x, model, device):
    model.eval()
    x = x.to(device)
    with torch.no_grad():
        out = model(x)
    return out.squeeze().cpu().numpy()
df["encoded"] = df["preprocessed"].apply(lambda x: encoder(x, model, device))

Encode SMILES of chromophores using pre-trained chemBERTa RNN

In [30]:
#assignment of canonical SMILES of ligand
smiles_dict = {
    "NRQ": r"CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CRQ": r"NC(=O)CCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "NRP": r"CC(C)CC(=N)C1=NC(=C/c2ccc(O)cc2)/C(=O)N1CC(O)=O",
    "CH6": r"CSCC[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CRO": r"[C@@H](O)[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "5SQ": r"N[C@@H](Cc1c[nH]cn1)C2=NC(=C\c3ccc(O)cc3)/C(=O)N2CC(O)=O",
    "4M9": r"NC(=O)CCC(=N)C1=NC(=C\c2c[nH]c3ccccc23)/C(=O)N1CC(O)=O",
    "CR2": r"NCC1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "OFM": r"C[C@H]1O[C@@](O)(N=C1C2=N\C(=C/c3ccc(O)cc3)C(=O)N2CC(O)=O)[C@@H](N)Cc4ccccc4",
    "CR8": r"N[C@@H](Cc1[nH]cnc1)c2nc(C=C3C=CC(=O)C=C3)c([O-])n2CC(O)=O",
    "CFY": r"N[C@@H](Cc1ccccc1)[C@@]2(O)SCC(=N2)C3=NC(=C\c4ccc(O)cc4)/C(=O)N3CC(O)=O",
    "OIM": r"CC[C@H](C)[C@H](N)[C@@]1(O)O[C@H](C)C(=N1)C2=N\C(=C/c3ccc(O)cc3)C(=O)N2CC(O)=O",
    "CH7": r"OC(=O)CN1C(=O)C(=C/c2ccc(O)cc2)/N=C1C3=NCCCC3",
    "GYS": r"N[C@@H](CO)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "WCR": r"C[C@@]1(O)NC(=C\c2ccc(O)cc2)/C(=O)N1CC(O)=O",
    "DYG": r"N[C@@H](CC(O)=O)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "FAD": r"Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P@](O)(=O)O[P@@](O)(=O)OC[C@H]4O[C@H]([C@H](O)[C@@H]4O)n5cnc6c(N)ncnc56)c2cc1C",
    "PIA": r"C[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "BLR": r"CC1=C(C=C)C(/NC1=O)=C/c2[nH]c(Cc3[nH]c(\C=C4/NC(=O)C(=C4C)C=C)c(C)c3CCC(O)=O)c(CCC(O)=O)c2C",
    "CRF": r"C[C@@H](O)[C@H](N)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1CC(O)=O",
    "NYG": r"N[C@@H](CC(N)=O)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "FMN": r"Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P](O)(O)=O)c2cc1C",
    "B2H": r"C[C@@H](O)[C@H](N)c1nc(Cc2c[nH]c3ccccc23)c(O)n1CC(O)=O",
    "SWG": r"N[C@@H](CO)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1CC(O)=O",
    "CSH": r"N[C@@H](CO)[C@H]1N[C@@H](Cc2c[nH]cn2)C(=O)N1CC(O)=O",
    "BJF": r"CC(C)C[C@H](N)c1nc(CC(C)C)c(O)n1CC(O)=O",
    "GYC": r"N[C@@H](CS)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CCY": r"N[C@@H](CS)[C@H]1N[C@@H](Cc2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CR7": r"NCCCC[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O"
}   
df["smiles"] = df["Chromophore/ligand"].str.strip().str.upper().map(smiles_dict)
from transformers import AutoTokenizer, AutoModel


model_name = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def smiles_to_vector(smiles):
    if not isinstance(smiles, str):
        return None

    inputs = tokenizer(smiles, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    # CLS token embedding = molecular representation
    pooled = outputs.last_hidden_state.max(dim=1).values.squeeze()

    return pooled.numpy()


df["smiles_vectors"] = df["smiles"].apply(smiles_to_vector)

Now, the vectors that result from encoding are concatenated with other descriptors

In [31]:
def concat(row):
    return np.concatenate([
        row["encoded"],                    
        row["smiles_vectors"],         
        np.array([
            row["Stokes shift"],     
            row["kDa"]
        ])
    ])

Normalization and dataset split:

In [32]:
scaler = StandardScaler()
num_cols = ["Stokes shift", "kDa"]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols]  = scaler.transform(test_df[num_cols])

Construction of input vectors

In [33]:
train_df["inputs_model1"] = train_df.apply(concat, axis=1)
test_df["inputs_model1"]  = test_df.apply(concat, axis=1)
